# Lesson 26 — Regex: Practice (SUPPORTING)

Work in `lessons/lesson-26-regex/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Write each deliverable in its stated file (R16 — no
`solution.py`, no double-write), then run each **Driver:** cell. Every
`Expected` block must match byte-exact.

You create: `patterns.py` (E1–M2) · `extract.py` + `scan.py` (Challenge,
which imports `patterns.py` — write once, reuse).
Provided: `buggy_scan.py` (buggy — true hunt, you fix it in place).
Installs: none — stdlib only (`re` taught in lesson.md).


## E1 — Ask the right question per call

## Intent
**Compile one pattern and use the correct question — anywhere, at start, whole line.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `patterns.py`, define module-scope `PING = re.compile(r"ping (\d+)")`
plus `found_anywhere(line)`, `found_at_start(line)`, `fully_matches(line)`,
each returning the captured id string or `None`. Use `search` / `match` /
`fullmatch` respectively — the question is the deliverable.

**File tree**

- `patterns.py` (`PING`, three functions)

**Interface contract**

- Captured id returned as text (no conversion here); miss returns `None`.

**Constraints**

- **One compiled pattern, three distinct calls.**
- **Raw string for the pattern.**

**Coverage**

- Compile + three questions → lesson.md Core Concept 1.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
import importlib
import patterns as p; importlib.reload(p)
assert p.found_anywhere("xx ping 3 yy") == "3"
assert p.found_anywhere("ping 3") == "3"
assert p.found_anywhere("pong") is None
assert p.found_at_start("ping 3 yy") == "3"
assert p.found_at_start("xx ping 3") is None
assert p.fully_matches("ping 3") == "3"
assert p.fully_matches("ping 3!") is None
assert p.fully_matches("xx ping 3") is None
print("E1 OK")

## E2 — Capture fields and harvest all matches

## Intent
**Extract every incident id from a block — captures out, offsets never written.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `patterns.py`, define `ERROR_IDS = re.compile(r"error (\d+)")` plus
`all_ids(block)` returning the id-text list via `findall`, and
`iter_ids(block)` yielding id text via `finditer` (lazy generator —
pulls flow on demand).

**File tree**

- `patterns.py` (`ERROR_IDS`, `all_ids`, `iter_ids`)

**Interface contract**

- `all_ids` returns `list[str]`; `iter_ids` returns an iterator of `str`.

**Constraints**

- **`findall` for the list, `finditer` for the stream — no hand loops.**
- **No offset bookkeeping anywhere.**

**Coverage**

- Harvest loops → lesson.md Core Concept 5.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
import types
import importlib
import patterns as p; importlib.reload(p)

assert p.all_ids("error 1 ok error 2") == ["1", "2"]
assert p.all_ids("all clear") == []
g = p.iter_ids("error 1 ok error 2")
assert isinstance(g, types.GeneratorType)
assert next(g) == "1"
assert list(g) == ["2"]
print("E2 OK")

## E3 — Flags plus an anchored gate

## Intent
**Match both casings across lines — and reject rot at the boundary.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `patterns.py`, define `ANY_ERROR = re.compile(r"^error (\d+)", re.IGNORECASE | re.MULTILINE)`
plus `count_errors(block)` (number of matching lines via `findall`) and
`parse_line(line)` (anchored gate: `fullmatch` on stripped text against
`^(error|skip) (\d+): (.+)$`, case-insensitive; return
`(kind.lower(), int(id), message)`; miss raises
`ValueError("invalid line: bad shape")`).

**File tree**

- `patterns.py` (`ANY_ERROR`, `count_errors`, `parse_line`)

**Interface contract**

- `count_errors` returns `int`; `parse_line` returns the triple or raises.

**Constraints**

- **Flags at compile time, stated visibly.**
- **Gate uses `fullmatch`; exact error string verbatim.**

**Coverage**

- Flags + validation gate → lesson.md Core Concepts 4/6.

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
import importlib
import patterns as p; importlib.reload(p)

assert p.count_errors("ERROR 1\nok\nError 2") == 2
assert p.count_errors("all clear") == 0
assert p.parse_line("Error 12: disk full") == ("error", 12, "disk full")
assert p.parse_line("  SKIP 3: reboot note  ") == ("skip", 3, "reboot note")
for bad in ["alert system rebooted", "error 1", ""]:
    try:
        p.parse_line(bad)
    except ValueError as e:
        assert str(e) == "invalid line: bad shape"
    else:
        raise AssertionError(("no raise", bad))
print("E3 OK")

## M1 — Classify lines with conversions

## Intent
**Turn raw log lines into typed records — shape-validated, converted at entry.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `patterns.py`, define `classify(line)` using anchored compiled
patterns: `error N: message` → `("error", N, message)`;
`temp V` (signed int-or-float via `-?\d+(?:\.\d+)?`) → `("temp", float,
None)`; anything else → `ValueError("invalid line: bad shape")`.
Non-capturing group for the decimal part (taught in lesson.md), so
`group(1)` stays the value.

**File tree**

- `patterns.py` (`classify` — no new file)

**Interface contract**

- `classify("error 7: disk full") == ("error", 7, "disk full")`.
- `classify("temp -4.5") == ("temp", -4.5, None)`.

**Constraints**

- **Anchored patterns only (no `search`-as-gate).**
- **Convert at the boundary (`int`/`float` on entry).**
- **Exact error string verbatim.**

**Coverage**

- Groups + conversion → lesson.md Core Concepts 2/6.

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
import importlib
import patterns as p; importlib.reload(p)

assert p.classify("error 7: disk full") == ("error", 7, "disk full")
assert p.classify("ERROR 7: disk full") == ("error", 7, "disk full")
assert p.classify("temp 41.5") == ("temp", 41.5, None)
assert p.classify("temp -4") == ("temp", -4.0, None)
for bad in ["alert system rebooted", "temp xx", ""]:
    try:
        p.classify(bad)
    except ValueError as e:
        assert str(e) == "invalid line: bad shape"
    else:
        raise AssertionError(("no raise", bad))
print("M1 OK")

## M2 — Compose extract-then-report yourself

## Intent
**Build the incident pipeline from your own patterns — order observable.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `patterns.py`, define `summarize(block)` from YOUR `parse_line`
(this composition is the exercise — the driver checks behavior only).
Split the block into lines; parse each; keep `error` records as
`(id, message)` pairs in order; count `skip` records; drop nothing
silently — unparseable lines raise the gate error. Return
`(errors, skipped)` where `errors` is the pair list.

**File tree**

- `patterns.py` (`summarize` — no new file)

**Constraints**

- **Reuse your own gate (no reimplemented validation).**
- **Order preserved; counts derived from kept records.**

**Coverage**

- Learner-authored composition → M2 law + Core Concepts 2/6.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
import importlib
import patterns as p; importlib.reload(p)

errs, skipped = p.summarize("error 1: disk full\nskip 2: reboot note\nerror 3: net down")
assert errs == [(1, "disk full"), (3, "net down")], errs
assert skipped == 1
assert p.summarize("") == ([], 0)
try:
    p.summarize("alert system rebooted")
except ValueError as e:
    assert str(e) == "invalid line: bad shape"
else:
    raise AssertionError("no raise")
print("M2 OK")

## H1 — Hunt: four pattern bugs in buggy_scan

## Intent
**Diagnose four real regex failures in a provided module — fix each where it lives.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_scan.py` is provided, buggy: four planted failures, one per
Pitfall (greedy overcapture, `match`-as-gate, unescaped dot, missing
flag). Nothing else in the file is wrong. Run the DEBUGGING protocol:
reproduce via the driver, read the traceback, locate, hypothesize, fix,
explain. Fix in place; do not restructure the module.

**File tree**

- `buggy_scan.py` (`first_pair`, `is_ping`, `has_version`, `count_errors`
  — YOU fix all four)

**Interface contract** (fixed states, exact)

- `first_pair` returns the first `(id, message)` pair, never an
  overcapture across two incidents.
- `is_ping("ping 3")` is True; mid-line and trailed forms are False.
- `has_version` matches literal dots only.
- `count_errors` sees both casings.

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer.**
- **Fill one 7-row chain per bug in the next cell, as you fix.**

**Coverage**

- The four pattern failures → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1 OK
```


### H1 chain reports (one per bug — fill as you fix)

#### Bug 1 — first_pair overcapture
| Step | Content |
|---|---|
| Symptom | assert b.first_pair("error 1: x, error 2: y") == ("1", "x") |
| Trigger | first_pair |
| Hypothesis | greedy capture |
| Evidence | use (.+) |
| Cause | greedy pattern |
| Fix | use [^,]+ |
| Proof | green |

#### Bug 2 — is_ping gate
| Step | Content |
|---|---|
| Symptom | assert b.is_ping("ping 3!") is False |
| Trigger | is_ping |
| Hypothesis | fullmatch |
| Evidence | use match |
| Cause | use match instead of fullmatch |
| Fix | change to fullmatch |
| Proof | green |

#### Bug 3 — has_version dot
| Step | Content |
|---|---|
| Symptom | assert b.has_version("v1x2 ready") is False |
| Trigger | has_version |
| Hypothesis | the escape char |
| Evidence | raw . |
| Cause | . not used with escape \ |
| Fix | add backslash |
| Proof | green |

#### Bug 4 — count_errors flag
| Step | Content |
|---|---|
| Symptom | assert b.count_errors("ERROR 1\nok\nError 2") == 2 |
| Trigger | count_errors |
| Hypothesis | case handling |
| Evidence | no ignore case |
| Cause | no re.IGNORECASE in re.complile |
| Fix | add re.IGNORECASE |
| Proof | green |


In [ ]:
import importlib
import buggy_scan as b
importlib.reload(b)

assert b.first_pair("error 1: x, error 2: y") == ("1", "x")
assert b.first_pair("error 9: solo") == ("9", "solo")
assert b.is_ping("ping 3") is True
assert b.is_ping("xx ping 3") is False
assert b.is_ping("ping 3!") is False
assert b.has_version("v1.2 ready") is True
assert b.has_version("v1x2 ready") is False
assert b.count_errors("ERROR 1\nok\nError 2") == 2
print("H1 OK")

## Challenge — Dawn extract run over the service log

## Intent
**Ship an extract run over the service log: every incident captured, every rotten line skipped loudly — shape-enforced end to end.**

**Type:** REAL-WORLD — Micro Build — writes 2 files

**Task**
The engineer runs one command before dawn:
`python scan.py <service.log> <incidents.txt>`. It classifies each log
line by shape, collects errors with ids, skips rotten lines with a loud
`skip` line, and writes an incidents report that is also printed — same
bytes both places.

Create `extract.py` and `scan.py` in this folder. Reuse `patterns.py`
(`classify` — write once, import, never double-write).

**Flow** (the whole wiring — every arrow is required)

```text
main reads <service.log> as text lines (always a valid text file; small log, read whole)
  → per (i, raw) in file order:
      kind, num, msg = classify(raw)      # rotten → ValueError → skip line
      error → "error {i}: {num}: {msg}" line (file index, then record id)
      temp over 40.0 → "alert {i}: {num}" line
      valid cool temps + skip records → "skip {i}: invalid line: bad shape" line
      temp over 40.0 → "alert {i}: {num}" line
      valid cool temps + skip records → skip line with the gate message
  → text = build_extract(lines)
  → write text to <incidents.txt> AND print the same text to stdout
```

The log is always a valid text file — no corrupt-file handling required.
"Missing log" means the path does not exist or is not a file.

**File tree**

- `extract.py` (`build_extract`)
- `scan.py` (`main`)

**Interface contract**

- `extract.build_extract(lines)` returns header + lines + trailing
  newline (header alone when empty). It performs no I/O.
- Header shape: `errors E, skipped S` (counts derived from line
  prefixes `error ` / `skip `; alert lines count as errors).
- `scan.main(argv)` — house shape (L20–L25): full `sys.argv`
  (`main(sys.argv)` guard), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Per line: classify → error/alert/skip routing; valid cool temps
  emit a skip line carrying the gate message shape.**
- **Alert threshold is exactly `40.0` (strictly over).**
- **Wrong argc prints `usage: scan.py <service.log> <incidents.txt>`
  to stderr and exits 1.**
- **Missing log prints `bad log: not found` to stderr and exits 1.**
- **Stdout and the incidents file are byte-identical.**

**Coverage**

- Patterns + classification + house entry → lesson.md Core Concepts
  1–6 + house entry (L20–L25 retrieval).

**Expected** (Driver cell, real subprocess runs, fixture of 6):
```text
CH OK
```


In [ ]:
import subprocess, sys, tempfile, pathlib
HOME = pathlib.Path.cwd()
log_lines = ["error 1: disk full", "temp 41.5", "temp 20.0", "oops", "ERROR 2: net down", "temp xx"]
expected = ("errors 3, skipped 3\n" "error 0: 1: disk full\n"
    "alert 1: 41.5\n" "skip 2: invalid line: bad shape\n" "skip 3: invalid line: bad shape\n"
    "error 4: 2: net down\n" "skip 5: invalid line: bad shape\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    lg, rp = td / "service.log", td / "incidents.txt"
    lg.write_text("\n".join(log_lines) + "\n", encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "scan.py", str(lg), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    usage = subprocess.run(
        [sys.executable, "scan.py", str(lg)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: scan.py <service.log> <incidents.txt>\n"
    missing = subprocess.run(
        [sys.executable, "scan.py", str(td / "nope.log"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad log: not found\n"
print("CH OK")